# Example: Processing Lidar Data Directly with Databricks

> Use some Spatial-Utils functions to process points data in a LAZ dataset.

__Notes__

* This example was run using DBR 16.0 on a 3 worker cluster (each with 8 CPUs and 61GB RAM);
* Uses 'spatial-utils-v1' branch with extra packages [route,viz].
* Requires a cluster with product `ST_` spatial sql functions enabled for KeplerGL Viz (may require photon cluster + spatial sql flag enabled).
* For the notebook to render well in github, we add screenshots of the map rendering and charts as well as artificially limit tabular results; when you run the notebook in databricks, you can uncomment and remove limits if desired.
* Requires you to download the [Lidar USA](https://www.lidarusa.com/sample-data.html) and upload this to a Volume. 

---
__Author:__ Mathieu Pelletier <mathieu.pelletier@databricks.com> | _Last Modified:_ 04 APR 2025

### Limitations
- LAS Specification Version 1.4
- Point Data Record Format 3
- Support one file at a time
- Scaled value (x, y, z)
- Header/VLRS not returned

## Setup

In [0]:
%pip install -U "databricks-spatial[points,viz] @ git+https://github.com/mathieupelletier-db/mosaic.git@lidar-demo"

In [0]:
# Uninstall previous versions
# %pip uninstall --yes databricks-spatial


In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "mpelletier")
dbutils.widgets.text("database", "geospatial")
dbutils.widgets.text("table", "lidar")
dbutils.widgets.text("volume", "lidar")

In [0]:
catalog = dbutils.widgets.get("catalog")
database = dbutils.widgets.get("database")
table = dbutils.widgets.get("table")
volume = dbutils.widgets.get("volume")

In [0]:
%sql
-- CHANGE THESE VARIABLES AS NEEDED
USE CATALOG ${catalog};
CREATE DATABASE IF NOT EXISTS ${database};
CREATE VOLUME IF NOT EXISTS ${database}.${volume};

## Download example
Rural transmission lines collected with an HD32 mounted to a DJI M600.

![Transmission Lines](https://www.lidarusa.com/uploads/5/4/1/5/54154851/transmission_1_orig.png)

In [0]:
import os

# Define Python variables
file_id = "1Axv8tXKEIVP9zvg91DvaCxNc6SkZukfM"
output_file = f"/Volumes/{catalog}/{database}/{volume}/Velodyne1_001.laz"

# Construct the shell command
shell_command = f"""
curl "https://drive.usercontent.google.com/download?id={file_id}&confirm=xxx" -o {output_file}
"""

# Execute the shell command
os.system(shell_command)

## Custom Spark datasource (format = "las") supports both las and laz file formats.

Options that can be used:
- path: las/laz file path (one file at a time)
- chunkSize: controls the batch size (number of points to read and process in memory)

In [0]:
# Import custom source
import spatial.points

In [0]:
df = spark.read.format("las").option("path", output_file).load()
df.display()

## Persist to table to leverage Delta performance optimizations (data skipping, table statistics, Liquid Clutering...)

In [0]:
# Persist in table with schema evolution
df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog}.{database}.{table}")

In [0]:
%sql
-- Cluster table using coordinates
ALTER TABLE ${catalog}.${database}.${table} CLUSTER BY (x, y, z);
OPTIMIZE ${catalog}.${database}.${table} FULL;

# Explore dataset


In [0]:
df = spark.read.table(f"{catalog}.{database}.{table}")

### Find boundaries

In [0]:
from pyspark.sql.functions import col, min, max

# Calculate min and max for x, y, z
min_max_values = df.select(
    min(col("x")).alias("x_min"),
    max(col("x")).alias("x_max"),
    min(col("y")).alias("y_min"),
    max(col("y")).alias("y_max"),
    min(col("z")).alias("z_min"),
    max(col("z")).alias("z_max")
)

display(min_max_values)

## Point Cloud Filtering

Possible values for classification

| Classification Value (bits 0:4) | Meaning                          |
|---------------------------------|----------------------------------|
| 0                               | Created, never classified        |
| 1                               | Unclassified1                    |
| 2                               | Ground                           |
| 3                               | Low Vegetation                   |
| 4                               | Medium Vegetation                |
| 5                               | High Vegetation                  |
| 6                               | Building                         |
| 7                               | Low Point (noise)                |
| 8                               | Model Key-point (mass point)     |
| 9                               | Water                            |
| 10                              | Reserved for ASPRS Definition    |
| 11                              | Reserved for ASPRS Definition    |
| 12                              | Overlap Points2                  |
| 13-31                           | Reserved for ASPRS Definition    |

In [0]:
%sql
-- Explore classification
SELECT DISTINCT(classification) FROM ${catalog}.${database}.${table}


In [0]:
%sql
-- Filter on user data
SELECT x,y,z,intensity FROM ${catalog}.${database}.${table}
WHERE user_data = 9

## Observe intensity distribution

In [0]:
%sql
SELECT intensity FROM ${catalog}.${database}.${table}


Databricks visualization. Run in Databricks to view.

# Vizualizations

### Use Plotly to show an interactive Scatter graph

In [0]:
from plotly.offline import init_notebook_mode, plot
import plotly.graph_objs as go

## Add bounding box to limit x,y,z points

In [0]:
#x_min	x_max	y_min	y_max	z_min	z_max
#2149298.2	2150688.2	1650235.6	1651142.8	591.093	725.273

X_min = 2150200
Y_min = 1650300
h = 500
w = 500

lidar_3d_sdf = (
  df
  .where(col("x").between(X_min, X_min + w - 1))
  .where(col("y").between(Y_min, Y_min + h - 1))
  .where(col("z").between(620, 720))
)

print(f"""count: {lidar_3d_sdf.count():,}""")

In [0]:
import numpy as np

las = lidar_3d_sdf.toPandas()

trace1 = go.Scatter3d(
  x=las.x, y=las.y, z=las.z, mode='markers',  
  marker=dict(size=2, color=las.intensity, colorscale='Viridis', opacity=1)
)

data = [trace1]
layout = go.Layout(
  autosize=True, width=1100, height=600,
  margin=dict(l=0, r=0, b=0, t=0),
  scene=dict(xaxis=dict(title="X"), yaxis=dict(title="Y"), zaxis=dict(title="Z"), aspectmode="data")
)

fig = go.Figure(data=data, layout=layout)
displayHTML(plot(fig, filename='3d-scatter-colorscale', output_type='div'))

## Sample data size to reduce the number of points to display (<400K)

In [0]:
from pyspark.sql.functions import col

# Define the fractions for each stratum
fractions = {0: 0.1, 1: 0.2, 2: 1.0}

# Apply sampleBy with the specified fractions
sampled_df = df.sample(False, 0.04, seed=42)

display(sampled_df.count())

In [0]:
import numpy as np

las = sampled_df.toPandas()

trace1 = go.Scatter3d(
  x=las.x, y=las.y, z=las.z, mode='markers',  
  marker=dict(size=2, color=las.intensity, colorscale='Viridis', opacity=1)
)

data = [trace1]
layout = go.Layout(
  autosize=True, width=1100, height=600,
  margin=dict(l=0, r=0, b=0, t=0),
  scene=dict(xaxis=dict(title="X"), yaxis=dict(title="Y"), zaxis=dict(title="Z"), aspectmode="data")
)

fig = go.Figure(data=data, layout=layout)
displayHTML(plot(fig, filename='3d-scatter-colorscale', output_type='div'))

## Use PIL to display all points (2D view only)

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.cm as cm

# Example DataFrame
las = df.sample(fraction=0.01).toPandas()  # Increase the sample fraction for more points

# Determine image size
x_max = 512
y_max = 512

# Normalize x and y coordinates to fit into the 512x512 image
las['x_normalized'] = ((las['x'] - las['x'].min()) / (las['x'].max() - las['x'].min()) * (x_max - 1)).astype(int)
las['y_normalized'] = ((las['y'] - las['y'].min()) / (las['y'].max() - las['y'].min()) * (y_max - 1)).astype(int)

# Create image array with white background
img = np.ones((y_max, x_max, 3), dtype=np.uint8) * 255

# Normalize intensity values to range [0, 1] for colormap
norm_intensity = (las['intensity'] - las['intensity'].min()) / (las['intensity'].max() - las['intensity'].min())

# Get colors from colormap
colors = cm.viridis(norm_intensity)

# Populate image array with colors
for i in range(len(las)):
    img[las['y_normalized'].iloc[i], las['x_normalized'].iloc[i]] = (colors[i][:3] * 255).astype(np.uint8)

# Display the image
#plt.imshow(img)
#plt.show()

# Save the image
img_pil = Image.fromarray(img)
display(img_pil)

## Display use matplotlib (non interactive)

In [0]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Create a figure
fig = plt.figure(figsize=(48, 24))

# 3D Scatter Plot
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
np.random.seed(0)

las = df.sample(fraction=0.2).toPandas()

x = las.x
y = las.y
z = las.z
intensity = las.intensity

# Use intensity for color mapping
scatter = ax1.scatter(x, y, z, c=intensity, cmap='viridis', s=1)
ax1.set_xlabel('X Axis')
ax1.set_ylabel('Y Axis')
ax1.set_zlabel('Z Axis')
ax1.set_title('3D Scatter Plot')

# Add color bar
cbar = fig.colorbar(scatter, ax=ax1, shrink=0.5, aspect=5)
cbar.set_label('Intensity')

# Change the viewing angle
ax1.view_init(elev=30, azim=45)  # Adjust the elevation and azimuth angles as needed

plt.show()